# 00. Завантаження бази ETER та відокремлення потрібних даних у `df_base`

У цьому ноутбуці створюється вибірка з потрібними даними та виконується очищення для подальшого аналізу.

Первинний набір даних: `data/raw/ETER_fullDump_27042023.csv`.

Результати:

- `data/processed/df_base.csv`
- `data/processed/df_base_clean.csv`
- `data/processed/special_codes.csv`


# 1. Етап завантаження та перевірки набору даних

Імпортування потрібних бібліотек і налаштування відображення табличних даних.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

Імпортування набору даних за допомогою `pathlib`.

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in {"notebooks", "scripts"} else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

path = DATA_DIR / "ETER_fullDump_27042023.csv"
df_raw = pd.read_csv(
    path,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
    low_memory=False,
)

FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


NOTEBOOK_PREFIX = "00_"


def save_figure(filename, fig=None, dpi=200):
    normalized_name = filename if filename.startswith(NOTEBOOK_PREFIX) else f"{NOTEBOOK_PREFIX}{filename}"
    path = FIGURES_DIR / normalized_name
    figure = fig or plt.gcf()
    figure.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


Первинна перевірка набору даних

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29591 entries, 0 to 29590
Columns: 1011 entries, ETER ID Year to Percentage of students with disability status at ISCED 6, categorised
dtypes: float64(12), int64(1), object(998)
memory usage: 228.2+ MB


In [4]:
df_raw.shape

(29591, 1011)

In [5]:
df_raw.head(5)

,ETER ID Year,ETER ID,National identifier,ROR ID,WHED ID,Institution Name,English Institution Name,Reference year,Institution Acronym,Country Code,Legal status,Notes on institution name,Institution Category - National Language,Institution Category - English,Institution Category standardised,Notes on institution category,Foundation year,Notes on foundation year,Legal status year,Notes on legal status year,Ancestor year,Notes on ancestor year,University hospital,University hospital OrgReg ID,University hospital name,Notes on university hospital,Member of European University alliance,European University Alliance OrgReg ID,European University Alliance name,European University Alliance acronym,Institutional website,Notes on institutional website,Region of establishment (NUTS 2),Region of establishment (NUTS 3),Sub-Country,Notes on region of establishment,Name of the city,Geographic coordinates - latitude,Geographic coordinates - longitude,Postcode,Multi-site institution,NUTS 3 codes of other campuses,Multi-site institution - City,Multi-site institution - Latitude,Multi-site institution - Longitude,Multi-site institution - Postcode,Notes on multisite institution,Personnel expenditure,Personnel expenditure.1,Personnel expenditure.2,Flag Personnel expenditure,Non-personnel expenditure,Non-personnel expenditure.1,Non-personnel expenditure.2,Flag Non-personnel expenditure,Expenditure unclassified,Expenditure unclassified.1,Expenditure unclassified.2,Flag Expenditure unclassified,Total Current expenditure,Total Current expenditure.1,Total Current expenditure.2,Flag Total current expenditure,Capital expenditure,Capital expenditure.1,Capital expenditure.2,Flag Capital expenditure,Accounting system of capital expenditure,Notes on expenditure,Basic government allocation,...,Co-publications outside EU-27 - Arts and Humanities,Co-publications outside EU-27 - Social Sciences,"Co-publications outside EU-27 - Business, administration and law",Co-publications outside EU-27 - Natural Sciences,Co-publications outside EU-27 - Information and Communication Technologies,Co-publications outside EU-27 - Engineering and Manufacturing,"Co-publications outside EU-27 - Agriculture, Forestry",Co-publications outside EU-27 - Health and Welfare,Flag on publication data,Notes on publication data,Dropout rate ISCED 6,Graduation on time ISCED 6,Graduation on time ISCED 7,Graduate unemployment ISCED 6,Graduate unemployment ISCED 7,Pedagogically skilled teaching staff,Teaching staff trained in technology-enabled learning,Embeddedness of microcredentials,Art related outputs,Professional publications per academic personnel,Income from continous professional development,Graduate companies,Spin-offs,External research revenues from regional sources,Student internships in the region,Bachelor graduates working in the region,Master graduates working in the region,Relative costs of joint degree programmes,Share of students in joint degree programmes,Share of joint degree programmes,Share of bachelor programmes taught in foreign language,Digital investment,Green courses,Green monitoring,Share of online degree programmes,Number of outreach programmes,Share of students with children ISCED 6,Share of students with disability status ISCED 6,"Senior academic personnel / academic staff (HC), categorised","Total graduates (ISCED5-7), categorised","Master degree orientation, categorised","Percentage of graduates in STEM, categorised","Student subject concentration, categorised","Dropout rate at ISCED 6, categorised","Embeddedness of microcredentials, categorised","Share of publications in 10% top-cited, categorised","Professional publications per academic personnel, categorised","PhD students per academic personnel, categorised","Publications per academic personnel, categorised","EU-FP project cooperation intensity with industrial partners, categorised","Spin-offs, categorised","Academic patent intensity, categorised","Share of copublications with industry partners, categorised","Sha

In [6]:
df_raw.columns.tolist()

['ETER ID Year',
 'ETER ID',
 'National identifier',
 'ROR ID',
 'WHED ID',
 'Institution Name',
 'English Institution Name',
 'Reference year',
 'Institution Acronym',
 'Country Code',
 'Legal status',
 'Notes on institution name',
 'Institution Category - National Language',
 'Institution Category - English',
 'Institution Category standardised',
 'Notes on institution category',
 'Foundation year',
 'Notes on foundation year',
 'Legal status year',
 'Notes on legal status year',
 'Ancestor year',
 'Notes on ancestor year',
 'University hospital',
 'University hospital OrgReg ID',
 'University hospital name',
 'Notes on university hospital',
 'Member of European University alliance',
 'European University Alliance OrgReg ID',
 'European University Alliance name',
 'European University Alliance acronym',
 'Institutional website',
 'Notes on institutional website',
 'Region of establishment (NUTS 2)',
 'Region of establishment (NUTS 3)',
 'Sub-Country',
 'Notes on region of establishme

In [7]:
df_raw.dtypes.value_counts()

object     998
float64     12
int64        1
Name: count, dtype: int64

# 2. Створення потрібної вибірки та перетворення її на датафрейм

In [8]:
selected_columns = [
    # IDs
    "ETER ID Year",
    "ETER ID",
    "Institution Name",
    "English Institution Name",
    "Reference year",

    # Geography
    "Country Code",
    "Region of establishment (NUTS 2)",
    "Region of establishment (NUTS 3)",
    "Name of the city",
    "Geographic coordinates - latitude",
    "Geographic coordinates - longitude",
    "Multi-site institution",

    # Institution type
    "Legal status",
    "Institution Category - English",
    "Institution Category standardised",

    # Students
    "Total students enrolled at ISCED 5",
    "Total students enrolled at ISCED 6",
    "Total students enrolled at ISCED 7",
    "Total students enrolled ISCED 7 long degree",
    "Total students enrolled ISCED 5-7",
    "Total students enrolled at ISCED 8",

    # Foreign students
    "Students enrolled at ISCED 5 - foreigner",
    "Students enrolled at ISCED 6 - foreigner",
    "Students enrolled at ISCED 7 - foreigner",
    "Students enrolled ISCED 7 long degree - foreigner",
    "Students enrolled at ISCED 5-7 - foreigner",
    "Students enrolled at ISCED 8 - foreigner",

    # Staff
    "Total academic personnel (FTE)",
    "Total personnel (FTE)",
    "Number of support and administrative personnel (FTE)",

    # Finance
    "Total Current expenditure.1",
    "Total Current expenditure.2",
    "Total Current revenues.1",
    "Total Current revenues.2",
    "Total third party funding.1",
    "Total third party funding.2",
    "R&D Expenditure.1",
    "R&D Expenditure.2",

    # Quality flags: students totals
    "Flag Total students ISCED 5",
    "Flag Total students ISCED 6",
    "Flag Total students ISCED 7",
    "Flag Total students ISCED 7 long degree",
    "Flag Total students ISCED 5-7",
    "Flag Total students ISCED 8",

    # Quality flags: foreign students / citizenship breakdowns
    "Flag Students ISCED 5 - citizenship",
    "Flag Students ISCED 6 - citizenship",
    "Flag Students ISCED 7 - citizenship",
    "Flag Students ISCED 7 long degree - citizenship",
    "Flag students enrolled at ISCED 5-7 - citizenship",
    "Flag Students ISCED 8 - citizenship",

    # Quality flags: staff
    "Flag Total academic personnel (FTE)",
    "Flag Total personnel (FTE)",
    "Flag Number of support and administrative personnel (FTE)",

    # Quality flags: finance
    "Flag Total current expenditure",
    "Flag Total current revenues",
    "Flag Total third party funding",
    "Flag R&D Expenditure",
]

rename_map = {
    "ETER ID Year": "record_id",
    "ETER ID": "institution_id",
    "Institution Name": "institution_name",
    "English Institution Name": "institution_name_en",
    "Reference year": "year",
    "Country Code": "country_code",

    "Region of establishment (NUTS 2)": "nuts2",
    "Region of establishment (NUTS 3)": "nuts3",
    "Name of the city": "city",
    "Geographic coordinates - latitude": "latitude",
    "Geographic coordinates - longitude": "longitude",
    "Multi-site institution": "multi_site",

    "Legal status": "legal_status",
    "Institution Category - English": "institution_category_en",
    "Institution Category standardised": "institution_category_std",

    "Total students enrolled at ISCED 5": "students_isced5",
    "Total students enrolled at ISCED 6": "students_isced6",
    "Total students enrolled at ISCED 7": "students_isced7",
    "Total students enrolled ISCED 7 long degree": "students_isced7_long_degree",
    "Total students enrolled ISCED 5-7": "students_isced5_7_total",
    "Total students enrolled at ISCED 8": "students_isced8",

    "Students enrolled at ISCED 5 - foreigner": "foreign_students_isced5",
    "Students enrolled at ISCED 6 - foreigner": "foreign_students_isced6",
    "Students enrolled at ISCED 7 - foreigner": "foreign_students_isced7",
    "Students enrolled ISCED 7 long degree - foreigner": "foreign_students_isced7_long_degree",
    "Students enrolled at ISCED 5-7 - foreigner": "foreign_students_isced5_7_total",
    "Students enrolled at ISCED 8 - foreigner": "foreign_students_isced8",

    "Total academic personnel (FTE)": "academic_personnel_fte",
    "Total personnel (FTE)": "total_personnel_fte",
    "Number of support and administrative personnel (FTE)": "support_admin_personnel_fte",

    "Total Current expenditure.1": "total_current_expenditure_eur",
    "Total Current expenditure.2": "total_current_expenditure_ppp",
    "Total Current revenues.1": "total_current_revenues_eur",
    "Total Current revenues.2": "total_current_revenues_ppp",
    "Total third party funding.1": "third_party_funding_eur",
    "Total third party funding.2": "third_party_funding_ppp",
    "R&D Expenditure.1": "rd_expenditure_eur",
    "R&D Expenditure.2": "rd_expenditure_ppp",

    "Flag Total students ISCED 5": "flag_students_isced5",
    "Flag Total students ISCED 6": "flag_students_isced6",
    "Flag Total students ISCED 7": "flag_students_isced7",
    "Flag Total students ISCED 7 long degree": "flag_students_isced7_long_degree",
    "Flag Total students ISCED 5-7": "flag_students_isced5_7_total",
    "Flag Total students ISCED 8": "flag_students_isced8",

    "Flag Students ISCED 5 - citizenship": "flag_foreign_students_isced5",
    "Flag Students ISCED 6 - citizenship": "flag_foreign_students_isced6",
    "Flag Students ISCED 7 - citizenship": "flag_foreign_students_isced7",
    "Flag Students ISCED 7 long degree - citizenship": "flag_foreign_students_isced7_long_degree",
    "Flag students enrolled at ISCED 5-7 - citizenship": "flag_foreign_students_isced5_7_total",
    "Flag Students ISCED 8 - citizenship": "flag_foreign_students_isced8",

    "Flag Total academic personnel (FTE)": "flag_academic_personnel_fte",
    "Flag Total personnel (FTE)": "flag_total_personnel_fte",
    "Flag Number of support and administrative personnel (FTE)": "flag_support_admin_personnel_fte",

    "Flag Total current expenditure": "flag_total_current_expenditure",
    "Flag Total current revenues": "flag_total_current_revenues",
    "Flag Total third party funding": "flag_third_party_funding",
    "Flag R&D Expenditure": "flag_rd_expenditure",
}

Перевірка наявності вибраних колонок

In [9]:
existing_columns = [col for col in selected_columns if col in df_raw.columns]
missing_selected_columns = [col for col in selected_columns if col not in df_raw.columns]

missing_selected_columns

[]

Створення таблиці з вибірки

In [10]:
df_base = df_raw[existing_columns].copy()
df_base = df_base.rename(columns=rename_map)
df_base.shape

(29591, 57)

# 3. Перегляд та перевірка таблиці df_base

In [11]:
df_base.head()

,record_id,institution_id,institution_name,institution_name_en,year,country_code,nuts2,nuts3,city,latitude,longitude,multi_site,legal_status,institution_category_en,institution_category_std,students_isced5,students_isced6,students_isced7,students_isced7_long_degree,students_isced5_7_total,students_isced8,foreign_students_isced5,foreign_students_isced6,foreign_students_isced7,foreign_students_isced7_long_degree,foreign_students_isced5_7_total,foreign_students_isced8,academic_personnel_fte,total_personnel_fte,support_admin_personnel_fte,total_current_expenditure_eur,total_current_expenditure_ppp,total_current_revenues_eur,total_current_revenues_ppp,third_party_funding_eur,third_party_funding_ppp,rd_expenditure_eur,rd_expenditure_ppp,flag_students_isced5,flag_students_isced6,flag_students_isced7,flag_students_isced7_long_degree,flag_students_isced5_7_total,flag_students_isced8,flag_foreign_students_isced5,flag_foreign_students_isced6,flag_foreign_students_isced7,flag_foreign_students_isced7_long_degree,flag_foreign_students_isced5_7_total,flag_foreign_students_isced8,flag_academic_personnel_fte,flag_total_personnel_fte,flag_support_admin_personnel_fte,flag_total_current_expenditure,flag_total_current_revenues,flag_third_party_funding,flag_rd_expenditure
0,AD0001.2023,AD0001,Universitat d'Andorra,University of Andorra,2023,AD,AD00,AD000,Sant Julià de Lòria,42.4632273,1.4865593,0,0,University,1,m,m,m,0,651,24,m,m,m,0,m,m,m,m,m,4488648,5038047.028452775,4488648,5038047.028452775,191909,215398.17049217128,m,m,NaN,NaN,NaN,NaN,i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d,NaN,NaN,NaN
1,AL0001.2023,AL0001,Universiteti i Tiranës,University of Tirana,2023,AL,AL02,AL022,Tirana,41.318093,19.821796,0,0,University,1,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AL0002.2023,AL0002,Universiteti Politeknik i Tiranës,Polytechnic University of Tirana,2023,AL,AL02,AL022,Tirana,41.316598,19.82149,0,0,University,1,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AL0003.2023,AL0003,Universiteti Bujqësor i Tiranës,Agricultural University of Tirana,2023,AL,AL02,AL022,Tirana,41.363,19.769534,0,0,University,1,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AL0004.2023,AL0004,"Universiteti i Elbasanit ""Aleksandër Xhuvani""","University of Elbasan, Aleksander Xhuvani",2023,AL,AL02,AL021,Elbasan,41.122408,20.079552,0,0,University,1,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29591 entries, 0 to 29590
Data columns (total 57 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   record_id                                 29591 non-null  object
 1   institution_id                            29591 non-null  object
 2   institution_name                          29591 non-null  object
 3   institution_name_en                       27016 non-null  object
 4   year                                      29591 non-null  int64 
 5   country_code                              29591 non-null  object
 6   nuts2                                     29591 non-null  object
 7   nuts3                                     29591 non-null  object
 8   city                                      29591 non-null  object
 9   latitude                                  29591 non-null  object
 10  longitude                                 2959

In [13]:
df_base.columns.tolist()

['record_id',
 'institution_id',
 'institution_name',
 'institution_name_en',
 'year',
 'country_code',
 'nuts2',
 'nuts3',
 'city',
 'latitude',
 'longitude',
 'multi_site',
 'legal_status',
 'institution_category_en',
 'institution_category_std',
 'students_isced5',
 'students_isced6',
 'students_isced7',
 'students_isced7_long_degree',
 'students_isced5_7_total',
 'students_isced8',
 'foreign_students_isced5',
 'foreign_students_isced6',
 'foreign_students_isced7',
 'foreign_students_isced7_long_degree',
 'foreign_students_isced5_7_total',
 'foreign_students_isced8',
 'academic_personnel_fte',
 'total_personnel_fte',
 'support_admin_personnel_fte',
 'total_current_expenditure_eur',
 'total_current_expenditure_ppp',
 'total_current_revenues_eur',
 'total_current_revenues_ppp',
 'third_party_funding_eur',
 'third_party_funding_ppp',
 'rd_expenditure_eur',
 'rd_expenditure_ppp',
 'flag_students_isced5',
 'flag_students_isced6',
 'flag_students_isced7',
 'flag_students_isced7_long_degre

In [14]:
df_base.dtypes

record_id                                   object
institution_id                              object
institution_name                            object
institution_name_en                         object
year                                         int64
country_code                                object
nuts2                                       object
nuts3                                       object
city                                        object
latitude                                    object
longitude                                   object
multi_site                                  object
legal_status                                object
institution_category_en                     object
institution_category_std                    object
students_isced5                             object
students_isced6                             object
students_isced7                             object
students_isced7_long_degree                 object
students_isced5_7_total        

In [15]:
df_base.dtypes.value_counts()

object    56
int64      1
Name: count, dtype: int64

Перевірка дублікатів у `df_base`

In [16]:
duplicates_report = pd.DataFrame({
    "check": [
        "full_row_duplicates",
        "record_id_duplicates",
        "institution_id_year_duplicates",
    ],
    "duplicates_count": [
        df_base.duplicated().sum(),
        df_base["record_id"].duplicated().sum(),
        df_base.duplicated(subset=["institution_id", "year"]).sum(),
    ],
})

duplicates_report

,check,duplicates_count
0,full_row_duplicates,0
1,record_id_duplicates,0
2,institution_id_year_duplicates,0


Перевірка пропусків у `df_base`

In [17]:
missing_report = pd.DataFrame({
    "missing_count": df_base.isna().sum(),
    "missing_rate": df_base.isna().mean()
})

missing_report = missing_report.sort_values("missing_count", ascending=False)

missing_report

,missing_count,missing_rate
flag_foreign_students_isced7_long_degree,28845,0.974790
flag_rd_expenditure,28587,0.966071
flag_foreign_students_isced6,28468,0.962049
flag_foreign_students_isced7,28453,0.961542
flag_third_party_funding,28405,0.959920
flag_foreign_students_isced5,28333,0.957487
flag_total_current_expenditure,28274,0.955493
flag_foreign_students_isced8,27927,0.943767
flag_foreign_students_isced5_7_total,27622,0.933459
flag_support_admin_personnel_fte,27454,0.927782


# 4. Робота зі спеціальними значеннями: пошук та заміна

До набору даних додаються метадані, у яких є розшифрування спеціальних значень.

In [18]:
special_codes = pd.read_excel(
    DATA_DIR / "CorrespondenceTable_NamesCodesToLabels.xlsx",
    sheet_name="SpecialCodes",
)

output_path = output_dir / "special_codes.csv"

special_codes.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

special_codes

,Special code,Label
0,a,not applicable
1,m,information missing
2,x,"breakdown not available, but included in total"
3,xc,included in another subcolumn
4,xr,included in another row
5,nc,"Data not collected, since variables was introd..."
6,c,confidential
7,s,value larger than 0 and below or equal to 3 re...


Порахуємо кількість і частку спеціальних кодів у кожному стовпці.

In [19]:
special_codes = ["a", "m", "x", "xc", "xr", "nc", "c", "s"]

special_code_report = pd.DataFrame({
    "special_code_count": df_base.astype("string").apply(
        lambda col: col.str.strip().isin(special_codes).sum()
    )
})

special_code_report["special_code_rate"] = (
    special_code_report["special_code_count"] / len(df_base)
)

special_code_report["special_code_pct"] = (
    special_code_report["special_code_rate"] * 100
)

special_code_report = special_code_report.sort_values(
    "special_code_count",
    ascending=False
)

special_code_report

,special_code_count,special_code_rate,special_code_pct
rd_expenditure_ppp,19291,0.651921,65.192119
rd_expenditure_eur,19291,0.651921,65.192119
third_party_funding_ppp,17490,0.591058,59.105809
third_party_funding_eur,17490,0.591058,59.105809
total_current_expenditure_ppp,16586,0.560508,56.050826
total_current_expenditure_eur,16586,0.560508,56.050826
total_current_revenues_ppp,16411,0.554594,55.459430
total_current_revenues_eur,16411,0.554594,55.459430
support_admin_personnel_fte,16304,0.550978,55.097834
total_personnel_fte,15873,0.536413,53.641310


Збереження df_base до заміни спеціальних значень

In [20]:
output_path = output_dir / "df_base.csv"

df_base.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


# 5. Заміна спеціальних значень

Заміна спеціальних значень у таблиці на NaN, окрім стовпців, назви яких починаються з "flag_".

In [21]:
flag_cols = [col for col in df_base.columns if col.startswith("flag_")]
non_flag_cols = [col for col in df_base.columns if col not in flag_cols]

df_base[non_flag_cols] = df_base[non_flag_cols].replace(
    r"^\s*(a|m|x|xc|xr|nc|c|s)\s*$",
    pd.NA,
    regex=True
)

Перевірка пропусків після заміни спеціальних значень на NA

In [22]:
missing_after_cleaning = pd.DataFrame({
    "missing_count": df_base.isna().sum(),
    "missing_rate": df_base.isna().mean()
}).sort_values("missing_count", ascending=False)

missing_after_cleaning

,missing_count,missing_rate
flag_foreign_students_isced7_long_degree,28845,0.974790
flag_rd_expenditure,28587,0.966071
flag_foreign_students_isced6,28468,0.962049
flag_foreign_students_isced7,28453,0.961542
flag_third_party_funding,28405,0.959920
flag_foreign_students_isced5,28333,0.957487
flag_total_current_expenditure,28274,0.955493
flag_foreign_students_isced8,27927,0.943767
flag_foreign_students_isced5_7_total,27622,0.933459
flag_support_admin_personnel_fte,27454,0.927782


Перевірка аналітичних колонок на наявність спеціальних значень

In [23]:
special_codes_left_non_flags = (
    df_base[non_flag_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().isin(special_codes).sum())
    .sort_values(ascending=False)
)

special_codes_left_non_flags[special_codes_left_non_flags > 0]

Series([], dtype: int64)

Перевірка прапорцевих колонок на наявність спеціальних значень

In [24]:
special_codes_in_flags = (
    df_base[flag_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().isin(special_codes).sum())
    .sort_values(ascending=False)
)

special_codes_in_flags[special_codes_in_flags > 0]

flag_rd_expenditure                         347
flag_third_party_funding                    344
flag_total_current_revenues                 344
flag_total_current_expenditure              330
flag_students_isced5_7_total                224
flag_students_isced6                        174
flag_students_isced7                        153
flag_students_isced7_long_degree            144
flag_students_isced5                        137
flag_academic_personnel_fte                 122
flag_total_personnel_fte                    122
flag_foreign_students_isced6                111
flag_students_isced8                        106
flag_foreign_students_isced5_7_total        101
flag_support_admin_personnel_fte             96
flag_foreign_students_isced7                 94
flag_foreign_students_isced7_long_degree     94
flag_foreign_students_isced8                 87
flag_foreign_students_isced5                 47
dtype: int64

Перетворення числових колонок з `str` на `float`

In [25]:
numeric_float_cols = [
    "students_isced5",
    "students_isced6",
    "students_isced7",
    "students_isced7_long_degree",
    "students_isced5_7_total",
    "students_isced8",

    "foreign_students_isced5",
    "foreign_students_isced6",
    "foreign_students_isced7",
    "foreign_students_isced7_long_degree",
    "foreign_students_isced5_7_total",
    "foreign_students_isced8",

    "academic_personnel_fte",
    "total_personnel_fte",
    "support_admin_personnel_fte",

    "total_current_expenditure_eur",
    "total_current_expenditure_ppp",
    "total_current_revenues_eur",
    "total_current_revenues_ppp",
    "third_party_funding_eur",
    "third_party_funding_ppp",
    "rd_expenditure_eur",
    "rd_expenditure_ppp",

    "latitude",
    "longitude",
]
df_base[numeric_float_cols] = (
    df_base[numeric_float_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.replace(",", ".", regex=False))
    .apply(pd.to_numeric, errors="coerce")
)

text_cols = df_base.select_dtypes(include=["object", "string"]).columns

df_base[text_cols] = (
    df_base[text_cols]
    .astype("string")
    .apply(lambda col: col.str.strip())
)

In [26]:
df_base.dtypes

record_id                                   string[python]
institution_id                              string[python]
institution_name                            string[python]
institution_name_en                         string[python]
year                                                 int64
country_code                                string[python]
nuts2                                       string[python]
nuts3                                       string[python]
city                                        string[python]
latitude                                           Float64
longitude                                          Float64
multi_site                                  string[python]
legal_status                                string[python]
institution_category_en                     string[python]
institution_category_std                    string[python]
students_isced5                                    Float64
students_isced6                                    Float

# 6. Перевірка на від'ємні значення

In [27]:
student_cols = [
    "students_isced5", "students_isced6", "students_isced7",
    "students_isced7_long_degree", "students_isced5_7_total",
    "students_isced8", "foreign_students_isced5",
    "foreign_students_isced6", "foreign_students_isced7",
    "foreign_students_isced7_long_degree",
    "foreign_students_isced5_7_total", "foreign_students_isced8",
]

staff_cols = [
    "academic_personnel_fte",
    "total_personnel_fte",
    "support_admin_personnel_fte",
]

finance_cols = [
    "total_current_expenditure_eur", "total_current_expenditure_ppp",
    "total_current_revenues_eur", "total_current_revenues_ppp",
    "third_party_funding_eur", "third_party_funding_ppp",
    "rd_expenditure_eur", "rd_expenditure_ppp",
]

check_cols = student_cols + staff_cols + finance_cols

anomaly_summary = pd.DataFrame({
    "negative_values": df_base[check_cols].lt(0).sum(),
    "missing_values": df_base[check_cols].isna().sum(),
})

anomaly_summary

,negative_values,missing_values
students_isced5,0,2787
students_isced6,0,2972
students_isced7,0,3138
students_isced7_long_degree,0,5655
students_isced5_7_total,0,3433
students_isced8,0,4620
foreign_students_isced5,0,7933
foreign_students_isced6,0,9024
foreign_students_isced7,0,8913
foreign_students_isced7_long_degree,0,9152


In [28]:
coordinate_anomalies = df_base[
    df_base["latitude"].notna()
    & df_base["longitude"].notna()
    & (
        ~df_base["latitude"].between(-90, 90)
        | ~df_base["longitude"].between(-180, 180)
    )
][[
    "record_id",
    "institution_id",
    "institution_name",
    "country_code",
    "city",
    "latitude",
    "longitude",
]]

coordinate_anomalies

,record_id,institution_id,institution_name,country_code,city,latitude,longitude


In [29]:
student_staff_diagnostic = df_base.assign(
    students_total_diagnostic=(
        df_base["students_isced5_7_total"].fillna(0)
        + df_base["students_isced8"].fillna(0)
    )
)

student_staff_diagnostic["student_staff_ratio_diagnostic"] = (
    student_staff_diagnostic["students_total_diagnostic"]
    / student_staff_diagnostic["academic_personnel_fte"]
)

student_staff_ratio_anomalies = student_staff_diagnostic[
    student_staff_diagnostic["academic_personnel_fte"].gt(0)
    & (
        student_staff_diagnostic["student_staff_ratio_diagnostic"].gt(100)
        | student_staff_diagnostic["student_staff_ratio_diagnostic"].lt(1)
    )
][[
    "record_id",
    "institution_id",
    "institution_name",
    "country_code",
    "year",
    "students_total_diagnostic",
    "academic_personnel_fte",
    "student_staff_ratio_diagnostic",
]]

student_staff_ratio_anomalies.sort_values(
    "student_staff_ratio_diagnostic",
    ascending=False
)

,record_id,institution_id,institution_name,country_code,year,students_total_diagnostic,academic_personnel_fte,student_staff_ratio_diagnostic
5743,PL0370.2022,PL0370,Wyższa Szkoła Nauk o Zdrowiu w Bydgoszczy,PL,2022,3235.0,5.0,647.0
9636,AT0075.2020,AT0075,Central European University Private University,AT,2020,922.333333,3.04,303.399123
12308,UK0253.2020,UK0253,Nelson College London Ltd,UK,2020,1465.0,5.0,293.0
2044,IT0048.2023,IT0048,Università Telematica PEGASO,IT,2023,118060.0,415.0,284.481928
5167,IT0048.2022,IT0048,Università Telematica PEGASO,IT,2022,82070.0,320.0,256.46875
...,...,...,...,...,...,...,...,...
12305,UK0250.2020,UK0250,Moorlands College,UK,2020,0.0,10.0,0.0
12309,UK0254.2020,UK0254,New College of the Humanities,UK,2020,0.0,50.0,0.0
12317,UK0264.2020,UK0264,Pearson College,UK,2020,0.0,35.0,0.0
12318,UK0265.2020,UK0265,Point Blank Music School,UK,2020,0.0,25.0,0.0


# 7. Збереження очищеної вибірки для подальшого аналізу

In [30]:
output_path = output_dir / "df_base_clean.csv"

df_base.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)